In [10]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_core.prompts import PromptTemplate, FewShotPromptTemplate
from dotenv import load_dotenv
import os
from openai import OpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.output_parsers import PydanticOutputParser
from langchain_classic.output_parsers import OutputFixingParser, BooleanOutputParser, DatetimeOutputParser
from langchain_core.exceptions import OutputParserException

In [11]:
load_dotenv()

True

In [12]:
api_key = os.getenv("GROQ_API_KEY")

In [13]:
llm= ChatOpenAI(
    model="openai/gpt-oss-20b",
    temperature=0.0,
    api_key=api_key,
    base_url="https://api.groq.com/openai/v1"
)

#llm.invoke----provide whole aimessage
#llm.invoke('dkja').content-----will provide only string output content

In [14]:
parser = StrOutputParser()

In [15]:
parser.invoke(
    llm.invoke('hello')
)
#will give only string values as output

'Hello! How can I help you today?'

date time parser

In [16]:
llm.invoke(
    'output todays date'
    'dont say anything'
)

AIMessage(content='2026-09-30', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 90, 'prompt_tokens': 77, 'total_tokens': 167, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 75, 'rejected_prediction_tokens': None, 'text_tokens': None}, 'prompt_tokens_details': None, 'queue_time': 0.314205846, 'prompt_time': 0.005208688, 'completion_time': 0.094847908, 'total_time': 0.100056596}, 'model_provider': 'openai', 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_9b8528b477', 'id': 'chatcmpl-9cb5d7bf-d3f5-4b64-8564-b9ca5c297901', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a0f37d-1854-7ac2-9248-0cdc6719ce31-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 77, 'output_tokens': 90, 'total_tokens': 167, 'input_token_details': {}, 'output_token_details': {'reasoning': 75}})

In [17]:
parser = DatetimeOutputParser()

In [18]:
parser.invoke(
    llm.invoke('output todays datetime %Y-%m-%dT%H:%M:%S.%fZ,dont say anything')
)

datetime.datetime(2026, 9, 30, 12, 34, 56, 789012)

boolean

In [19]:
parser = BooleanOutputParser()

In [20]:
parser.invoke(
    llm.invoke('are you an agent? yes or no')
)

False

structured

In [21]:
from typing_extensions import Annotated, TypedDict

In [22]:
class UserInfo(TypedDict):
    '''user info'''
    name: Annotated[str, "","user's name."]
    country: Annotated[str,"","where he lives"]

In [23]:
llm_with_structure = llm.with_structured_output(UserInfo)

In [24]:
llm_with_structure.invoke(
    'my name is xyz. and live in cuba'
)

{'name': 'xyz', 'country': 'cuba'}

In [25]:
from pydantic import BaseModel,Field

class PydanticUserInfo(BaseModel):
    '''user info'''
    name: Annotated[str, Field(description='user name', default=None)]
    country: Annotated[str, Field(description='user country', default=None)]

In [26]:
llm_with_structure= llm.with_structured_output(PydanticUserInfo)

In [27]:
struc_output = llm_with_structure.invoke('the sky is blue')

In [28]:
struc_output

PydanticUserInfo(name='', country='')

Dealing with Errors

In [29]:
class Performance(BaseModel):
    '''actor info'''
    name: Annotated[str, Field(description='actor name')]
    film_name: Annotated[list[str], Field(description='top 5 filmography')]

In [30]:
llm_with_structure=llm.with_structured_output(Performance)

In [31]:
response=llm_with_structure.invoke('give top 5 films of Allu arjun')

In [32]:
response.json()

C:\Users\sreeram poojitha\AppData\Local\Temp\ipykernel_14504\690762135.py:1: PydanticDeprecatedSince20: The `json` method is deprecated; use `model_dump_json` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  response.json()


'{"name":"Allu Arjun","film_name":["Arya","Desamuduru","Badrinath","Julayi","Sarrainodu"]}'

In [33]:
parser=PydanticOutputParser(pydantic_object=Performance)

In [34]:
parser.parse(response.json())

C:\Users\sreeram poojitha\AppData\Local\Temp\ipykernel_14504\2130552313.py:1: PydanticDeprecatedSince20: The `json` method is deprecated; use `model_dump_json` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  parser.parse(response.json())


Performance(name='Allu Arjun', film_name=['Arya', 'Desamuduru', 'Badrinath', 'Julayi', 'Sarrainodu'])

In [35]:
mismatched = "{'name':'allu arjun','films': ['avengers']}"

In [36]:
try:
    parser.parse(mismatched)
except OutputParserException as e:
    print(e)

Invalid json output: {'name':'allu arjun','films': ['avengers']}
For troubleshooting, visit: https://docs.langchain.com/oss/python/langchain/errors/OUTPUT_PARSING_FAILURE 


In [37]:
try:
    response=llm_with_structure.invoke('give top 5 films of AA')
    print(response)
except Exception as e:
    print(e)

Error code: 400 - {'error': {'message': "Failed to validate JSON. Please adjust your prompt. See 'failed_generation' for more details.", 'type': 'invalid_request_error', 'code': 'json_validate_failed', 'failed_generation': ''}}


error fixing

In [38]:
from langchain_classic.output_parsers import OutputFixingParser

In [39]:
new_parser = OutputFixingParser.from_llm(parser=parser, llm=llm)

In [41]:
new_parser.parse(mismatched)

Performance(name='Allu Arjun', film_name=['Pushpa', 'Ala Vaikunthapurramuloo', 'Sarrainodu', 'Vinaya Vidheya Rama', 'Ala Vaikunthapurramuloo'])